# OCR od kuchni — od klasycznych metod do modeli end-to-end

Notebook towarzyszący 45-minutowemu seminarium.

**Cel**: zrozumieć **jak naprawdę działa OCR** na każdym poziomie abstrakcji — od pikseli, przez ręcznie projektowane cechy, sieci konwolucyjne, aż po transformery.

## Mapa notebooka

| # | Sekcja | Co pokazuje | Czas wykonania |
|---|--------|-------------|----------------|
| 0 | Setup | Importy, flagi, sprawdzenie środowiska | <1 s |
| 1 | Dane testowe | Pobranie + generowanie obrazów (druk, szum, rotacja, paragon, tablica) | ~5 s |
| 2 | Preprocessing | Otsu vs adaptive vs Sauvola, deskew Hough, morfologia | ~3 s |
| 3 | Segmentacja | Profile projekcji, connected components, ekstrakcja znaków | ~2 s |
| 4 | Klasyczne ML | HOG + SVM/kNN na cyfrach, confusion matrix, błędy | ~10 s |
| 5 | CNN (LeNet) | Trening w PyTorch + wizualizacja filtrów i feature maps | ~1-2 min |
| 6 | Gotowe silniki | Tesseract, EasyOCR, TrOCR — porównanie outputów | ~30 s + pobieranie modeli |
| 7 | Porównanie | Tabela CER × silnik × obraz + wnioski | ~5 s |

## Główne pytanie seminarium

> **Czy w 2026 r. ma jeszcze sens rozumieć OCR od środka, czy wystarczy zawołać `model.recognize(image)`?**

Każda sekcja dorzuca cegiełkę do odpowiedzi.

## 0. Setup

Importy, flagi sterujące, deterministyczność. Po pierwszym uruchomieniu zobaczysz, które komponenty są dostępne (Tesseract / EasyOCR / TrOCR / CUDA).

In [ ]:
from __future__ import annotations

import io
import os
import sys
import time
import shutil
import random
import warnings
from pathlib import Path
from dataclasses import dataclass, field
from typing import Callable

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont, ImageFilter

warnings.filterwarnings("ignore")

# ====== FLAGI — wyłącz co cięższe jeśli nie potrzebujesz ======
ENABLE_CNN_TRAINING = True      # ~1-2 min na CPU
ENABLE_TESSERACT    = True      # wymaga binarki tesseract
ENABLE_EASYOCR      = True      # ~70 MB pobrania przy 1. uruchomieniu
ENABLE_TROCR        = True      # ~330 MB pobrania przy 1. uruchomieniu

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# ====== Sanity check ======
print(f"Python      : {sys.version.split()[0]}")
print(f"NumPy       : {np.__version__}")

try:
    import cv2
    print(f"OpenCV      : {cv2.__version__}")
except Exception as e:
    print(f"OpenCV      : BRAK ({e})")

try:
    import skimage
    print(f"scikit-image: {skimage.__version__}")
except Exception as e:
    print(f"scikit-image: BRAK ({e})")

try:
    import torch
    print(f"PyTorch     : {torch.__version__}  (CUDA: {torch.cuda.is_available()})")
except Exception as e:
    print(f"PyTorch     : BRAK ({e})")

# Tesseract — auto-detect binarki
if ENABLE_TESSERACT:
    try:
        import pytesseract
        # na RHEL zwykle jest w PATH; jeśli nie - ustaw ścieżkę ręcznie:
        # pytesseract.pytesseract.tesseract_cmd = "/usr/bin/tesseract"
        ver = pytesseract.get_tesseract_version()
        print(f"Tesseract   : {ver}")
    except Exception as e:
        print(f"Tesseract   : BRAK / nie skonfigurowany ({e})")
        ENABLE_TESSERACT = False


## 1. Dane testowe

Aby uczciwie porównywać metody, potrzebujemy **zróżnicowanego zestawu obrazów**:

| Obraz | Co testuje |
|-------|-----------|
| `clean_print.png` | druk idealny — baseline |
| `noisy_print.png` | szum gaussowski + rozmycie — odporność |
| `rotated_print.png` | rotacja 7° — potrzeba deskew |
| `low_contrast.png` | nierównomierne oświetlenie — adaptive thresholding |
| `multiline.png` | wiele linijek — segmentacja wierszy |
| `license_plate.png` | krótki napis na kolorowym tle — scene text |
| `skimage_page.png` | prawdziwa skanowana strona (skimage built-in) |

Każdy obraz **generujemy lokalnie z PIL** — dzięki temu znamy *ground truth* i możemy liczyć CER. Dodatkowo próbujemy pobrać 1-2 prawdziwe obrazy z publicznych repo (z fallbackiem).

In [ ]:
import requests
from skimage import data as skdata, io as skio

# ----- helper: znajdź pasujący font systemowy -----
def find_font(size: int = 32) -> ImageFont.FreeTypeFont:
    candidates = [
        "/usr/share/fonts/dejavu-sans-fonts/DejaVuSans.ttf",
        "/usr/share/fonts/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/liberation-sans/LiberationSans-Regular.ttf",
        "/usr/share/fonts/liberation/LiberationSans-Regular.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/google-noto/NotoSans-Regular.ttf",
    ]
    for path in candidates:
        if Path(path).exists():
            return ImageFont.truetype(path, size)
    return ImageFont.load_default()

# ----- generatory syntetycznych obrazów -----
def render_text(
    text: str,
    size: tuple[int, int] = (800, 200),
    font_size: int = 36,
    bg: int = 255,
    fg: int = 0,
    padding: int = 20,
) -> Image.Image:
    img = Image.new("L", size, color=bg)
    draw = ImageDraw.Draw(img)
    font = find_font(font_size)
    draw.multiline_text((padding, padding), text, font=font, fill=fg, spacing=8)
    return img

def add_gaussian_noise(img: Image.Image, sigma: float = 25.0) -> Image.Image:
    arr = np.array(img).astype(np.float32)
    arr += np.random.normal(0, sigma, arr.shape)
    arr = np.clip(arr, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)

def add_uneven_lighting(img: Image.Image, strength: float = 80.0) -> Image.Image:
    """Symuluje nierówne oświetlenie — gradient po przekątnej."""
    arr = np.array(img).astype(np.float32)
    h, w = arr.shape
    yy, xx = np.mgrid[0:h, 0:w]
    grad = (xx / w + yy / h) / 2.0           # 0..1
    arr = arr - strength * grad              # przyciemnij narastająco
    arr = np.clip(arr, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)

def make_license_plate(text: str = "RKR 12345") -> Image.Image:
    img = Image.new("RGB", (520, 110), color=(245, 240, 220))
    draw = ImageDraw.Draw(img)
    draw.rectangle([5, 5, 515, 105], outline=(0, 0, 0), width=4)
    font = find_font(72)
    draw.text((40, 12), text, font=font, fill=(0, 0, 0))
    return img.convert("L")

# ----- ground truth (do liczenia CER w sekcji 7) -----
GT: dict[str, str] = {}

clean_text = "The quick brown fox\njumps over the lazy dog\n1234567890"
clean = render_text(clean_text, size=(800, 260), font_size=36)
clean.save(DATA_DIR / "clean_print.png")
GT["clean_print"] = clean_text

noisy = add_gaussian_noise(clean, sigma=30)
noisy = noisy.filter(ImageFilter.GaussianBlur(radius=0.8))
noisy.save(DATA_DIR / "noisy_print.png")
GT["noisy_print"] = clean_text

rotated = clean.rotate(7, resample=Image.BICUBIC, fillcolor=255, expand=True)
rotated.save(DATA_DIR / "rotated_print.png")
GT["rotated_print"] = clean_text

low_contrast = add_uneven_lighting(clean, strength=110)
low_contrast.save(DATA_DIR / "low_contrast.png")
GT["low_contrast"] = clean_text

multiline_text = (
    "Lorem ipsum dolor sit amet,\n"
    "consectetur adipiscing elit.\n"
    "Sed do eiusmod tempor incididunt\n"
    "ut labore et dolore magna aliqua."
)
multiline = render_text(multiline_text, size=(700, 240), font_size=28)
multiline.save(DATA_DIR / "multiline.png")
GT["multiline"] = multiline_text

plate_text = "RKR 12345"
plate = make_license_plate(plate_text)
plate.save(DATA_DIR / "license_plate.png")
GT["license_plate"] = plate_text

# ----- prawdziwa zeskanowana strona ze skimage -----
page_arr = skdata.page()                      # uint8 grayscale
Image.fromarray(page_arr).save(DATA_DIR / "skimage_page.png")
# GT dla page nie znamy — pomijamy w CER

# ----- opcjonalna próba pobrania prawdziwego scene-text -----
def try_download(url: str, dst: Path) -> bool:
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200 and len(r.content) > 1000:
            dst.write_bytes(r.content)
            return True
    except Exception:
        pass
    return False

scene_url = "https://raw.githubusercontent.com/JaidedAI/EasyOCR/master/examples/english.png"
scene_path = DATA_DIR / "scene_text.png"
if try_download(scene_url, scene_path):
    print(f"Pobrano scene_text.png ({scene_path.stat().st_size // 1024} kB)")
else:
    print("Nie udało się pobrać scene_text.png — używam tylko obrazów syntetycznych.")

print(f"\nWygenerowano {len(GT)} obrazów z ground truth + page + (opc.) scene.")
for k, v in GT.items():
    print(f"  {k:15s}  GT: {v[:40]!r}{'...' if len(v) > 40 else ''}")


In [ ]:
# ----- podgląd całego datasetu -----
img_files = sorted(DATA_DIR.glob("*.png"))
cols = 3
rows = (len(img_files) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(15, 4 * rows))
axes = axes.flatten()
for ax, p in zip(axes, img_files):
    im = Image.open(p)
    ax.imshow(im, cmap="gray")
    ax.set_title(p.stem, fontsize=11)
    ax.axis("off")
for ax in axes[len(img_files):]:
    ax.axis("off")
plt.tight_layout()
plt.show()


## 2. Preprocessing klasyczny

Każdy klasyczny pipeline OCR zaczyna się od *przygotowania* obrazu. Pokazujemy 3 najważniejsze elementy:

1. **Binaryzacja** — Otsu (globalny próg) vs adaptive vs Sauvola (oba lokalne). Pokażemy, że na nierównym oświetleniu Otsu pada, a Sauvola wygrywa.
2. **Deskew** — wyrównanie pochylenia przez transformatę Hougha.
3. **Morfologia** — opening / closing do usuwania szumu i łączenia fragmentów znaków.

> **Pointa**: te kroki są deterministyczne, debugowalne i tłumaczalne. To największa przewaga klasyki nad black-boxem.

In [ ]:
# ===== 2a. Binaryzacja: Otsu vs Adaptive vs Sauvola =====
import cv2
from skimage.filters import threshold_otsu, threshold_sauvola, threshold_niblack

low = np.array(Image.open(DATA_DIR / "low_contrast.png").convert("L"))

# Globalny Otsu
thr_otsu = threshold_otsu(low)
bin_otsu = (low > thr_otsu).astype(np.uint8) * 255

# Adaptive (OpenCV, gaussian)
bin_adapt = cv2.adaptiveThreshold(
    low, 255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY,
    blockSize=35, C=10,
)

# Sauvola (skimage)
thr_sau = threshold_sauvola(low, window_size=25, k=0.2)
bin_sauvola = (low > thr_sau).astype(np.uint8) * 255

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].imshow(low, cmap="gray");        axes[0].set_title("Wejście (nierówne światło)")
axes[1].imshow(bin_otsu, cmap="gray");   axes[1].set_title(f"Otsu (próg={thr_otsu})")
axes[2].imshow(bin_adapt, cmap="gray");  axes[2].set_title("Adaptive Gaussian")
axes[3].imshow(bin_sauvola, cmap="gray");axes[3].set_title("Sauvola (lokalny)")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

print("Otsu zostawia czarne plamy w ciemniejszym rogu — globalny próg nie radzi sobie z gradientem oświetlenia.")
print("Sauvola/Adaptive liczą próg lokalnie i utrzymują czytelność tekstu w całym obrazie.")


In [ ]:
# ===== 2b. Deskew — wykrycie kąta przez Hougha =====
from skimage.transform import rotate as sk_rotate
from skimage.feature import canny
from skimage.transform import hough_line, hough_line_peaks

rot = np.array(Image.open(DATA_DIR / "rotated_print.png").convert("L"))

def estimate_skew_angle(gray: np.ndarray) -> float:
    edges = canny(gray, sigma=2.0)
    h, theta, d = hough_line(edges, theta=np.deg2rad(np.arange(-30, 30, 0.2)))
    _, angles, _ = hough_line_peaks(h, theta, d, num_peaks=20)
    if len(angles) == 0:
        return 0.0
    # linie tekstu są poziome -> theta blisko ±90°; przeliczamy na "ile odchylone od poziomu"
    skew_rad = np.median(angles) - np.pi / 2
    return float(np.rad2deg(skew_rad))

angle = estimate_skew_angle(rot)
deskewed = sk_rotate(rot, angle=angle, resize=True, cval=1.0, mode="constant")
deskewed = (deskewed * 255).astype(np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].imshow(rot, cmap="gray");       axes[0].set_title(f"Oryginał (wykryty kąt: {angle:+.2f}°)")
axes[1].imshow(deskewed, cmap="gray");  axes[1].set_title("Po deskew (rotacja kompensująca)")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()


In [ ]:
# ===== 2c. Morfologia — opening (usuwa kropki/szum) i closing (łączy fragmenty) =====
noisy = np.array(Image.open(DATA_DIR / "noisy_print.png").convert("L"))

# najpierw binaryzacja
_, bin_noisy = cv2.threshold(noisy, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
opened = cv2.morphologyEx(bin_noisy, cv2.MORPH_OPEN, kernel, iterations=1)
closed = cv2.morphologyEx(bin_noisy, cv2.MORPH_CLOSE, kernel, iterations=1)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].imshow(noisy, cmap="gray");   axes[0].set_title("Wejście (szum)")
axes[1].imshow(bin_noisy, cmap="gray"); axes[1].set_title("Po Otsu (inverted)")
axes[2].imshow(opened, cmap="gray");  axes[2].set_title("Opening — usuwa drobny szum")
axes[3].imshow(closed, cmap="gray");  axes[3].set_title("Closing — zamyka dziury w znakach")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()


## 3. Segmentacja klasyczna

W "starym" OCR rozpoznajemy **pojedyncze znaki**, więc trzeba je najpierw wyciąć z obrazu. Dwie najpopularniejsze techniki:

### 3a. Profile projekcji
Sumujemy piksele wzdłuż osi → minima w poziomej projekcji to **spacje między wierszami**, minima w pionowej projekcji to **spacje między znakami**.

### 3b. Connected Component Analysis (CCA)
Znajdujemy spójne grupy czarnych pikseli — każda taka grupa to potencjalny znak. Bardzo szybkie (jeden przelot przez obraz), ale wrażliwe na połączone znaki (np. kursywa) i znaki rozbite na części (np. "i" z kropką).

In [ ]:
# ===== 3a. Profile projekcji — wyodrębnij linijki =====
ml = np.array(Image.open(DATA_DIR / "multiline.png").convert("L"))
_, bin_ml = cv2.threshold(ml, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

# pozioma projekcja (suma wzdłuż osi X) — pokazuje gdzie są wiersze
h_proj = bin_ml.sum(axis=1)

# znajdź wiersze: regiony, gdzie h_proj > threshold
threshold = h_proj.max() * 0.05
in_line = h_proj > threshold
# znajdź "krawędzie" — start i koniec każdego wiersza
edges = np.diff(in_line.astype(int))
line_starts = np.where(edges == 1)[0]
line_ends   = np.where(edges == -1)[0]

fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [3, 1]})
axes[0].imshow(bin_ml, cmap="gray")
for s, e in zip(line_starts, line_ends):
    axes[0].axhline(s, color="lime", lw=1)
    axes[0].axhline(e, color="red",  lw=1)
axes[0].set_title(f"Wykryto {min(len(line_starts), len(line_ends))} wierszy")
axes[0].axis("off")

axes[1].plot(h_proj, np.arange(len(h_proj)))
axes[1].invert_yaxis()
axes[1].axvline(threshold, color="orange", linestyle="--", label="próg")
axes[1].set_title("Pozioma projekcja")
axes[1].set_xlabel("suma pikseli w wierszu")
axes[1].legend()
plt.tight_layout(); plt.show()


In [ ]:
# ===== 3b. Connected Components — wyciągnięcie pojedynczych znaków =====
clean = np.array(Image.open(DATA_DIR / "clean_print.png").convert("L"))
_, bin_clean = cv2.threshold(clean, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

num, labels, stats, centroids = cv2.connectedComponentsWithStats(bin_clean, connectivity=8)
# stats: [x, y, w, h, area]  (indeks 0 = tło)

# odfiltruj śmieci (zbyt małe / zbyt duże)
min_area, max_area = 20, bin_clean.size * 0.3
components = [
    (i, *stats[i])
    for i in range(1, num)
    if min_area <= stats[i, cv2.CC_STAT_AREA] <= max_area
]

# narysuj prostokąty
vis = cv2.cvtColor(clean, cv2.COLOR_GRAY2BGR)
for _, x, y, w, h, area in components:
    cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 200, 0), 1)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"Connected Components — {len(components)} kandydatów na znaki")
axes[0].axis("off")

# wyciągnij i pokaż pierwsze 30 znaków
crops = []
for _, x, y, w, h, _ in sorted(components, key=lambda c: (c[2], c[1]))[:30]:
    crop = bin_clean[y:y+h, x:x+w]
    pad = max(crop.shape) + 4
    canvas = np.zeros((pad, pad), dtype=np.uint8)
    oy, ox = (pad - crop.shape[0]) // 2, (pad - crop.shape[1]) // 2
    canvas[oy:oy+crop.shape[0], ox:ox+crop.shape[1]] = crop
    crops.append(canvas)

grid_rows, grid_cols = 3, 10
fig2, axes2 = plt.subplots(grid_rows, grid_cols, figsize=(12, 3.6))
for ax, c in zip(axes2.flatten(), crops):
    ax.imshow(c, cmap="gray"); ax.axis("off")
for ax in axes2.flatten()[len(crops):]: ax.axis("off")
plt.suptitle("Wyizolowane znaki (po normalizacji do kwadratu)")
plt.tight_layout(); plt.show()


## 4. Klasyczne ML — ręczne cechy + klasyfikator

Idea: zamiast podawać surowe piksele do klasyfikatora, projektujemy **deskryptor** odporny na małe odchylenia (przesunięcie, oświetlenie). Dwa klasyki:

- **HOG** (Histogram of Oriented Gradients) — siatka komórek, w każdej histogram orientacji gradientu. Świetnie radzi sobie z kształtami liter.
- Klasyfikator: **kNN** (najprostszy, dobry baseline) lub **SVM** z kernelem RBF.

Dataset: `sklearn.datasets.load_digits` (1797 cyfr 8×8) — uczy się w sekundy, idealne na demo. Po pokazaniu konceptu, na MNIST (sekcja 5) skok do CNN będzie bardziej naturalny.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from skimage.feature import hog
from skimage.transform import resize as sk_resize

digits = load_digits()
X_imgs = digits.images          # (1797, 8, 8)
y      = digits.target          # (1797,)
print(f"Liczba próbek: {len(y)},  klasy: {sorted(set(y))}")

# upscale do 32x32 — HOG potrzebuje większej rozdzielczości żeby mieć sensowne komórki
X_up = np.stack([sk_resize(im, (32, 32), anti_aliasing=True) for im in X_imgs])

def extract_hog(img: np.ndarray) -> np.ndarray:
    return hog(
        img,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        feature_vector=True,
    )

X_hog = np.array([extract_hog(im) for im in X_up])
X_pix = X_imgs.reshape(len(X_imgs), -1)          # baseline: surowe piksele
print(f"Wymiar wektora HOG: {X_hog.shape[1]},   wymiar surowych pikseli: {X_pix.shape[1]}")

# split + trening dwóch klasyfikatorów × dwie reprezentacje
X_tr_h, X_te_h, X_tr_p, X_te_p, y_tr, y_te = train_test_split(
    X_hog, X_pix, y, test_size=0.25, random_state=SEED, stratify=y
)

results = {}
for name, (X_tr, X_te) in {"HOG": (X_tr_h, X_te_h), "Piksele": (X_tr_p, X_te_p)}.items():
    for clf_name, clf in [("kNN(k=3)", KNeighborsClassifier(n_neighbors=3)),
                          ("SVM-RBF",  SVC(kernel="rbf", C=10, gamma="scale"))]:
        t0 = time.time()
        clf.fit(X_tr, y_tr)
        acc = accuracy_score(y_te, clf.predict(X_te))
        dt  = time.time() - t0
        results[f"{name} + {clf_name}"] = (acc, dt)

print("\n=== Wyniki na load_digits (test split 25%) ===")
print(f"{'Konfiguracja':28s} {'Accuracy':>10s} {'Czas':>10s}")
for k, (a, d) in results.items():
    print(f"{k:28s} {a:>10.4f} {d:>9.2f}s")


In [ ]:
# Confusion matrix + przykłady błędnie sklasyfikowane (HOG + SVM)
best_clf = SVC(kernel="rbf", C=10, gamma="scale").fit(X_tr_h, y_tr)
y_pred = best_clf.predict(X_te_h)
cm = confusion_matrix(y_te, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
im = axes[0].imshow(cm, cmap="Blues")
axes[0].set_title("Confusion matrix (HOG + SVM)")
axes[0].set_xlabel("Predykcja"); axes[0].set_ylabel("Prawda")
for i in range(10):
    for j in range(10):
        if cm[i, j] > 0:
            axes[0].text(j, i, cm[i, j], ha="center", va="center",
                         color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=8)
plt.colorbar(im, ax=axes[0], fraction=0.046)

# błędy
mistakes_idx = np.where(y_pred != y_te)[0]
axes[1].set_title(f"Błędy klasyfikacji ({len(mistakes_idx)} / {len(y_te)})")
axes[1].axis("off")
n_show = min(12, len(mistakes_idx))
if n_show > 0:
    grid = np.ones((4 * 32, 3 * 32))
    for k, idx in enumerate(mistakes_idx[:n_show]):
        r, c = k // 3, k % 3
        grid[r*32:(r+1)*32, c*32:(c+1)*32] = X_up[np.where(np.all(X_up == X_te_p[idx].reshape(8,8).repeat(4,0).repeat(4,1), axis=(1,2)))[0][0]] if False else 0
    # uproszczone: pokaż błędy w siatce z PIL
    fig2, ax_grid = plt.subplots(2, 6, figsize=(12, 4))
    # potrzebujemy oryginalnych obrazów z test setu — odzyskajmy je
    idx_in_full = np.array([np.where((X_pix == row).all(axis=1))[0][0] for row in X_te_p])
    for ax, idx in zip(ax_grid.flatten(), mistakes_idx[:12]):
        orig_idx = idx_in_full[idx]
        ax.imshow(X_imgs[orig_idx], cmap="gray")
        ax.set_title(f"GT:{y_te[idx]}→Pred:{y_pred[idx]}", fontsize=9)
        ax.axis("off")
    for ax in ax_grid.flatten()[n_show:]:
        ax.axis("off")
    plt.tight_layout()

plt.tight_layout(); plt.show()

print("\n=== Classification report ===")
print(classification_report(y_te, y_pred, digits=3))


In [ ]:
# Wizualizacja: jak wygląda sam HOG dla cyfry "5"
from skimage.feature import hog as hog_with_image

sample_idx = np.where(y == 5)[0][0]
sample = X_up[sample_idx]
features, hog_image = hog_with_image(
    sample,
    orientations=9,
    pixels_per_cell=(8, 8),
    cells_per_block=(2, 2),
    block_norm="L2-Hys",
    visualize=True,
)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(sample, cmap="gray"); axes[0].set_title("Cyfra '5' (32×32)"); axes[0].axis("off")
axes[1].imshow(hog_image, cmap="magma"); axes[1].set_title("Reprezentacja HOG"); axes[1].axis("off")
plt.tight_layout(); plt.show()

print("Każda 'gwiazdka' to histogram orientacji gradientu w komórce 8×8.")
print("HOG abstrahuje od bezwzględnej jasności — patrzy tylko na kierunki krawędzi.")


## 5. Mały CNN (LeNet) na MNIST

Pierwszy krok w stronę głębokiego uczenia: **sieć sama uczy się cech**, których w sekcji 4 musieliśmy projektować ręcznie.

Architektura LeNet-5 (1998, Yann LeCun) — historyczna, ale ciągle skuteczna:

```
Input(1,28,28) → Conv(6,5×5) → ReLU → MaxPool(2×2)
              → Conv(16,5×5) → ReLU → MaxPool(2×2)
              → FC(120) → ReLU → FC(84) → ReLU → FC(10)
```

Trening: 2 epoki na CPU = ~1-2 min. Po treningu obejrzymy filtry pierwszej warstwy i feature mapy — zobaczysz, że sieć "wymyśliła" sobie detektory krawędzi (podobne do tego, co w HOG projektowaliśmy ręcznie).

> Wyłącz `ENABLE_CNN_TRAINING = False` w sekcji 0 jeśli chcesz pominąć.

In [ ]:
if ENABLE_CNN_TRAINING:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader
    from torchvision import datasets, transforms

    torch.manual_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Trening na: {device}")

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
    ])

    train_ds = datasets.MNIST(root="data/mnist", train=True,  download=True, transform=transform)
    test_ds  = datasets.MNIST(root="data/mnist", train=False, download=True, transform=transform)
    train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=0)
    test_loader  = DataLoader(test_ds,  batch_size=512, shuffle=False, num_workers=0)
    print(f"MNIST: train={len(train_ds)}, test={len(test_ds)}")

    class LeNet5(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv1 = nn.Conv2d(1, 6, kernel_size=5, padding=2)   # 28x28 -> 28x28
            self.conv2 = nn.Conv2d(6, 16, kernel_size=5)             # 14x14 -> 10x10
            self.fc1   = nn.Linear(16 * 5 * 5, 120)
            self.fc2   = nn.Linear(120, 84)
            self.fc3   = nn.Linear(84, 10)
        def forward(self, x):
            x = F.max_pool2d(F.relu(self.conv1(x)), 2)               # -> 14x14
            x = F.max_pool2d(F.relu(self.conv2(x)), 2)               # -> 5x5
            x = x.flatten(1)
            x = F.relu(self.fc1(x))
            x = F.relu(self.fc2(x))
            return self.fc3(x)

    model = LeNet5().to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Parametry sieci: {n_params:,}  (~{n_params/1e3:.1f} k)")

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    EPOCHS = 2
    history = {"train_loss": [], "test_acc": []}
    for ep in range(EPOCHS):
        model.train()
        running = 0.0
        t0 = time.time()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            running += loss.item() * xb.size(0)
        avg_loss = running / len(train_ds)

        model.eval()
        correct = 0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb, yb = xb.to(device), yb.to(device)
                correct += (model(xb).argmax(1) == yb).sum().item()
        acc = correct / len(test_ds)
        history["train_loss"].append(avg_loss)
        history["test_acc"].append(acc)
        print(f"Epoka {ep+1}/{EPOCHS}  loss={avg_loss:.4f}  test_acc={acc:.4f}  ({time.time()-t0:.1f}s)")
else:
    print("Sekcja CNN pominięta (ENABLE_CNN_TRAINING = False).")


In [ ]:
if ENABLE_CNN_TRAINING:
    # ===== Wizualizacja: filtry conv1 (czego nauczyła się sieć?) =====
    filters = model.conv1.weight.detach().cpu().numpy()      # (6, 1, 5, 5)
    fig, axes = plt.subplots(1, 6, figsize=(12, 2.5))
    for i, ax in enumerate(axes):
        ax.imshow(filters[i, 0], cmap="gray")
        ax.set_title(f"filter {i}")
        ax.axis("off")
    plt.suptitle("Filtry pierwszej warstwy konwolucyjnej (5×5) — sieć sama wymyśliła detektory krawędzi")
    plt.tight_layout(); plt.show()

    # ===== Feature maps: jak sieć "widzi" cyfrę =====
    sample, label = test_ds[0]
    with torch.no_grad():
        x = sample.unsqueeze(0).to(device)
        fmap1 = F.relu(model.conv1(x)).cpu().numpy()[0]      # (6, 28, 28)
        fmap2 = F.relu(model.conv2(F.max_pool2d(F.relu(model.conv1(x)), 2))).cpu().numpy()[0]  # (16, 10, 10)

    fig, axes = plt.subplots(3, 6, figsize=(14, 7))
    axes[0, 0].imshow(sample.squeeze(), cmap="gray")
    axes[0, 0].set_title(f"Input (label={label})")
    axes[0, 0].axis("off")
    for j in range(1, 6):
        axes[0, j].axis("off")
    for i in range(6):
        axes[1, i].imshow(fmap1[i], cmap="viridis")
        axes[1, i].set_title(f"conv1 ch {i}")
        axes[1, i].axis("off")
    for i in range(6):
        axes[2, i].imshow(fmap2[i], cmap="viridis")
        axes[2, i].set_title(f"conv2 ch {i}")
        axes[2, i].axis("off")
    plt.suptitle("Feature maps — coraz bardziej abstrakcyjne reprezentacje w głębi sieci")
    plt.tight_layout(); plt.show()


## 6. Gotowe silniki OCR

Trzy reprezentanci trzech generacji:

| Silnik | Pod spodem | Typ |
|--------|-----------|-----|
| **Tesseract 5** | LSTM (z post-processingiem opartym na słownikach) | Klasyczny / hybrydowy |
| **EasyOCR** | Detektor CRAFT + recognizer CRNN+CTC (ResNet+BiLSTM+CTC) | Pełny DL |
| **TrOCR** | ViT encoder + RoBERTa-like decoder, koder-dekoder Transformerowy | End-to-end Transformer |

Wszystkie puszczamy na **tych samych obrazach** z sekcji 1. Outputy zbieramy do tabeli, którą wykorzystamy w sekcji 7 do liczenia CER.

In [ ]:
# Wspólna lista obrazów do testów (te z GT — żeby liczyć CER)
TEST_IMAGES: list[tuple[str, Path]] = [
    (name, DATA_DIR / f"{name}.png") for name in GT.keys()
]
# + page (bez GT, do oceny jakościowej)
TEST_IMAGES.append(("skimage_page", DATA_DIR / "skimage_page.png"))

# Tu zbieramy wyniki: {engine: {image_name: predicted_text}}
predictions: dict[str, dict[str, str]] = {}
timings:     dict[str, dict[str, float]] = {}


In [ ]:
# ===== 6a. Tesseract =====
if ENABLE_TESSERACT:
    import pytesseract
    predictions["Tesseract"] = {}
    timings["Tesseract"]     = {}
    for name, path in TEST_IMAGES:
        img = Image.open(path).convert("L")
        t0 = time.time()
        text = pytesseract.image_to_string(img, lang="eng", config="--psm 6").strip()
        timings["Tesseract"][name] = time.time() - t0
        predictions["Tesseract"][name] = text
        print(f"--- {name} ({timings['Tesseract'][name]*1000:.0f} ms) ---")
        print(text)
        print()
else:
    print("Tesseract wyłączony.")


In [ ]:
# ===== 6b. EasyOCR =====
if ENABLE_EASYOCR:
    try:
        import easyocr
        print("Ładuję modele EasyOCR (przy 1. uruchomieniu pobierze ~70 MB)…")
        reader = easyocr.Reader(["en"], gpu=False, verbose=False)
        predictions["EasyOCR"] = {}
        timings["EasyOCR"]     = {}
        for name, path in TEST_IMAGES:
            t0 = time.time()
            result = reader.readtext(str(path), detail=0, paragraph=True)
            timings["EasyOCR"][name] = time.time() - t0
            text = "\n".join(result).strip()
            predictions["EasyOCR"][name] = text
            print(f"--- {name} ({timings['EasyOCR'][name]*1000:.0f} ms) ---")
            print(text)
            print()
    except Exception as e:
        print(f"EasyOCR nie wystartował: {e}")
        ENABLE_EASYOCR = False
else:
    print("EasyOCR wyłączony.")


In [ ]:
# ===== 6c. TrOCR =====
# UWAGA: TrOCR rozpoznaje pojedyncze LINIE tekstu (model wyjściowy ma ograniczoną dł.).
# Dla obrazów wielo-liniowych musimy najpierw zsegmentować linie (sekcja 3a),
# potem każdą linijkę przepuścić oddzielnie i połączyć.

if ENABLE_TROCR:
    try:
        from transformers import TrOCRProcessor, VisionEncoderDecoderModel
        import torch as _t

        print("Ładuję microsoft/trocr-base-printed (przy 1. uruchomieniu ~330 MB)…")
        processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed")
        trocr_model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-printed")
        trocr_device = "cuda" if _t.cuda.is_available() else "cpu"
        trocr_model = trocr_model.to(trocr_device).eval()
        print(f"TrOCR na {trocr_device}")

        def trocr_line(pil_img: Image.Image) -> str:
            inputs = processor(images=pil_img.convert("RGB"), return_tensors="pt").pixel_values
            inputs = inputs.to(trocr_device)
            with _t.no_grad():
                ids = trocr_model.generate(inputs, max_new_tokens=64)
            return processor.batch_decode(ids, skip_special_tokens=True)[0]

        def segment_lines(pil_img: Image.Image) -> list[Image.Image]:
            """Prosta segmentacja linii przez poziomą projekcję (z sekcji 3)."""
            gray = np.array(pil_img.convert("L"))
            _, bw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
            proj = bw.sum(axis=1)
            mask = proj > proj.max() * 0.05
            edges = np.diff(mask.astype(int))
            starts = list(np.where(edges == 1)[0])
            ends   = list(np.where(edges == -1)[0])
            if mask[0]:  starts = [0] + starts
            if mask[-1]: ends   = ends + [len(mask)]
            lines = []
            for s, e in zip(starts, ends):
                if e - s < 5:
                    continue
                pad = 4
                s2 = max(0, s - pad); e2 = min(gray.shape[0], e + pad)
                lines.append(pil_img.crop((0, s2, pil_img.width, e2)))
            return lines or [pil_img]

        predictions["TrOCR"] = {}
        timings["TrOCR"]     = {}
        for name, path in TEST_IMAGES:
            img = Image.open(path).convert("L")
            t0 = time.time()
            lines = segment_lines(img)
            texts = [trocr_line(ln) for ln in lines]
            timings["TrOCR"][name] = time.time() - t0
            text = "\n".join(texts).strip()
            predictions["TrOCR"][name] = text
            print(f"--- {name} ({len(lines)} linii, {timings['TrOCR'][name]*1000:.0f} ms) ---")
            print(text)
            print()
    except Exception as e:
        print(f"TrOCR nie wystartował: {e}")
        ENABLE_TROCR = False
else:
    print("TrOCR wyłączony.")


## 7. Porównanie ilościowe — CER + czas

**CER (Character Error Rate)** = (substytucje + wstawienia + usunięcia) / długość referencji.
Liczone na bazie odległości Levenshteina, biblioteka `jiwer`.

Im niżej, tym lepiej. CER = 0.05 oznacza, że ~5% znaków jest błędnych.

In [ ]:
from jiwer import cer

# tylko obrazy z ground truth
gt_names = list(GT.keys())
engines  = list(predictions.keys())

# tabela CER
print(f"\n{'Obraz':18s} | " + " | ".join(f"{e:>12s}" for e in engines))
print("-" * (20 + 15 * len(engines)))
cer_matrix = {}
for name in gt_names:
    cer_matrix[name] = {}
    row_str = f"{name:18s} | "
    for e in engines:
        pred = predictions[e].get(name, "")
        score = cer(GT[name], pred)
        cer_matrix[name][e] = score
        row_str += f"{score:>12.3f} | "
    print(row_str)

# średnie po wszystkich obrazach
print("-" * (20 + 15 * len(engines)))
avg_str = f"{'ŚREDNIA CER':18s} | "
for e in engines:
    mean = np.mean([cer_matrix[n][e] for n in gt_names])
    avg_str += f"{mean:>12.3f} | "
print(avg_str)

# średnie czasy
if timings:
    print()
    avg_str = f"{'ŚREDNI CZAS [s]':18s} | "
    for e in engines:
        mean_t = np.mean(list(timings.get(e, {}).values())) if timings.get(e) else float("nan")
        avg_str += f"{mean_t:>12.2f} | "
    print(avg_str)


In [ ]:
# Wizualizacja: heatmapa CER
import matplotlib.colors as mcolors

if engines:
    M = np.array([[cer_matrix[n][e] for e in engines] for n in gt_names])

    fig, ax = plt.subplots(figsize=(2 + 1.6 * len(engines), 1 + 0.55 * len(gt_names)))
    im = ax.imshow(M, cmap="RdYlGn_r", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(engines))); ax.set_xticklabels(engines, rotation=20)
    ax.set_yticks(range(len(gt_names))); ax.set_yticklabels(gt_names)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, f"{M[i,j]:.2f}", ha="center", va="center",
                    color="white" if M[i,j] > 0.5 else "black", fontsize=10)
    ax.set_title("CER (niżej = lepiej)")
    plt.colorbar(im, ax=ax, fraction=0.04)
    plt.tight_layout(); plt.show()


In [ ]:
# Porównanie jakościowe: oryginał + predykcje obok siebie
for name in gt_names:
    print(f"\n{'='*70}")
    print(f"Obraz: {name}")
    print(f"{'='*70}")
    print(f"GT:\n{GT[name]}\n")
    for e in engines:
        pred = predictions[e].get(name, "")
        print(f"--- {e} (CER={cer_matrix[name][e]:.3f}) ---")
        print(pred)
        print()


## 8. Wnioski

Wracamy do pytania tytułowego:

> **Czy ma sens rozumieć OCR od środka, czy wystarczą black-boxy?**

### Co pokazały nasze eksperymenty

1. **Klasyczny pipeline jest interpretowalny.** Otsu vs Sauvola vs Adaptive — różnice widać gołym okiem, każdą decyzję możesz wytłumaczyć. W systemach krytycznych (medycyna, prawo, finanse) to bezcenne.
2. **HOG + SVM osiąga >97% na cyfrach** i robi to w ułamkach sekundy bez GPU. Dla wąskich domen (tablice rejestracyjne, formularze ze stałym układem) klasyka ciągle bije DL pod względem latencji i kosztu.
3. **CNN sam uczy się tych samych cech** co my projektujemy ręcznie (krawędzie, narożniki), tylko lepiej dopasowanych do zadania.
4. **TrOCR i EasyOCR** generalizują na rzeczy, których klasyka nie ogarnie (rotacje, fonty, niski kontrast), ale halucynują na trudnych przypadkach i są wolniejsze.

### Kiedy wybierać co

| Scenariusz | Najlepszy wybór | Dlaczego |
|------------|-----------------|----------|
| Tablice rejestracyjne, embedded | Klasyka (HOG+SVM) lub mały CNN | Niska latencja, deterministyczne |
| Skany dokumentów drukowanych | Tesseract / PaddleOCR | Sprawdzone, offline |
| Tekst w scenach | CRNN/PARSeq + detektor (DBNet) | SOTA na ICDAR |
| Faktury, formularze (KIE) | Donut / LayoutLMv3 / cloud | Rozumie układ |
| Pismo odręczne, manuskrypty | TrOCR-handwritten, VLM | Generalizacja |
| Prototyp z małą ilością danych | VLM (GPT-4o, Claude, Qwen-VL) | Zero-shot |
| Audytowalne, krytyczne | Klasyka + reguły + walidacja | Interpretowalność |

### Najważniejsza myśl

> **Black-box to nie zwolnienie z myślenia.** Bez znajomości fundamentów nie zdiagnozujesz, dlaczego model myli "0" z "O" na fakturze, ani kiedy preprocessing (deskew, binaryzacja) podniesie skuteczność TrOCR z 60% na 95% praktycznie za darmo.

---

### Co warto pokazać na seminarium ponad ten notebook

- **CAPTCHA jako case study** — wojna OCR vs anti-OCR.
- **Manga-OCR / Transkribus** — egzotyczne domeny.
- **Mathpix / pix2tex** — 2D OCR matematyki.
- **VLM jako OCR** (GPT-4o, Claude Vision, Qwen2-VL) — pokaż halucynacje na celowo zmanipulowanym paragonie.
- **Adversarial OCR** — minimalna perturbacja, która łamie CRNN, ale dla człowieka jest niewidoczna.

### Bibliografia (do `.tex`)

- Shi, Bai, Yao (2017) — *CRNN: End-to-End Trainable Neural Network for Image-Based Sequence Recognition*
- Graves et al. (2006) — *Connectionist Temporal Classification*
- Li et al. (2021) — *TrOCR: Transformer-based Optical Character Recognition with Pre-trained Models*
- Kim et al. (2022) — *Donut: OCR-free Document Understanding Transformer*
- Smith (2007) — *An Overview of the Tesseract OCR Engine*
- Bautista & Atienza (2022) — *PARSeq: Scene Text Recognition with Permuted Autoregressive Sequence Models*
- Liao et al. (2020) — *DBNet: Real-Time Scene Text Detection with Differentiable Binarization*